# LLAMA

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from sklearn.metrics import classification_report

import pandas as pd
from collections import defaultdict

from sklearn.metrics import accuracy_score, f1_score

from datasets import load_dataset

In [ ]:
dataset_hf = load_dataset("Fatima0923/Automated-Personality-Prediction")
print(dataset_hf)
print(dataset_hf["train"].column_names)

# load pandora dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['text', 'agreeableness', 'openness', 'conscientiousness', 'extraversion', 'neuroticism'],
        num_rows: 16047
    })
    validation: Dataset({
        features: ['text', 'agreeableness', 'openness', 'conscientiousness', 'extraversion', 'neuroticism'],
        num_rows: 2415
    })
    test: Dataset({
        features: ['text', 'agreeableness', 'openness', 'conscientiousness', 'extraversion', 'neuroticism'],
        num_rows: 2415
    })
})
['text', 'agreeableness', 'openness', 'conscientiousness', 'extraversion', 'neuroticism']


In [ ]:
def binarize(x, threshold=0.0):
    return 1 if x > threshold else 0

# one personality vector per user, taking first one

In [ ]:
def convert_dataset(hf_split):
    dataset = []

    for row in hf_split:
        dataset.append({
            "comments": [row["text"]],  # already aggregated
            "labels": {
                "Openness": binarize(row["openness"]),
                "Conscientiousness": binarize(row["conscientiousness"]),
                "Extraversion": binarize(row["extraversion"]),
                "Agreeableness": binarize(row["agreeableness"]),
                "Neuroticism": binarize(row["neuroticism"]),
            }
        })

    return dataset

# convert dataset

In [ ]:
dataset = convert_dataset(dataset_hf["test"])

# building dataset

In [ ]:
model_name = "NousResearch/Meta-Llama-3-8B-Instruct"

#LLAMA Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# tokenizer/model

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

In [ ]:
traits = [
    "Openness",
    "Conscientiousness",
    "Extraversion",
    "Agreeableness",
    "Neuroticism"
]

# traits from pandora database

In [ ]:
def build_simple_prompt(text, trait):
    return f"""
You are a psychological text classifier.

Your task is to determine whether the Big Five trait \"{trait}\" is expressed in the TEXT.

Rules:
- Output ONLY \"1\" if the trait is present
- Output ONLY \"0\" if the trait is not present
- Do NOT explain
- Do NOT output anything else

TEXT:
{text}
"""

# simple  prompt

In [ ]:
def build_enriched_prompt(text, trait):
    trait_descriptions = {
        "Openness": "curiosity, imagination, preference for novelty",
        "Conscientiousness": "organization, discipline, reliability",
        "Extraversion": "sociability, assertiveness, energy",
        "Agreeableness": "empathy, cooperation, kindness",
        "Neuroticism": "emotional instability, anxiety, mood swings"
    }

    return f"""
You are an expert in personality psychology using the Big Five model.

Determine whether the trait \"{trait}\" is present in the TEXT.

Trait definition:
{trait_descriptions[trait]}

Guidelines:
- Look at tone, wording, emotional expression, and behavior
- Consider both explicit and implicit signals

Output:
- \"1\" if present
- \"0\" if absent
- Output ONLY one digit

TEXT:
{text}
"""

# complex prompt

In [ ]:
def predict(text, trait, enriched=True):
    prompt = build_enriched_prompt(text, trait) if enriched else build_simple_prompt(text, trait)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False,
        temperature=0
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return parse_output(decoded[len(prompt):])

  # inference function

In [ ]:
def parse_output(output):
    output = output.strip()
    if output.startswith("1"):
        return 1
    elif output.startswith("0"):
        return 0
    return None


# output parsing

In [ ]:
def aggregate_user_comments(user_comments, max_chars=5000):
    text = " ".join(user_comments)
    return text[:max_chars]


# aggregate user text

In [ ]:
def run_evaluation(dataset, enriched=True):
    results = {}

    for trait in traits:
        y_true = []
        y_pred = []

        for user in dataset:
            text = aggregate_user_comments(user["comments"])

            pred = predict(text, trait, enriched=enriched)

            if pred is None:
                continue

            y_true.append(user["labels"][trait])
            y_pred.append(pred)

        print(f"\n===== {trait} =====")
        print(classification_report(y_true, y_pred, digits=3))

        results[trait] = (y_true, y_pred)

    return results

  # dataset loop

In [ ]:
results = run_evaluation(dataset, enriched=True)

#results of evaluation

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [ ]:
for trait in results:
    y_true, y_pred = results[trait]

    print(f"\n{trait}")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("F1:", f1_score(y_true, y_pred))

# metrics

# Mistral

In [ ]:
mistral_model_name = "mistralai/Mistral-7B-Instruct-v0.3"

# mistral model

In [ ]:
mistral_tokenizer = AutoTokenizer.from_pretrained(mistral_model_name)
mistral_model = AutoModelForCausalLM.from_pretrained(
    mistral_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# mistral tokenizer/model

In [ ]:
def build_mistral_simple_prompt(text, trait):
    return [
        {
            "role": "user",
            "content": (
                f'You are a psychological text classifier.\n\n'
                f'Your task is to determine whether the Big Five trait "{trait}" is expressed in the TEXT.\n\n'
                f'Rules:\n'
                f'- Output ONLY "1" if the trait is present\n'
                f'- Output ONLY "0" if the trait is not present\n'
                f'- Do NOT explain\n'
                f'- Do NOT output anything else\n\n'
                f'TEXT:\n{text}'
            )
        }
    ]

# simple prompt

In [ ]:
def build_mistral_enriched_prompt(text, trait):
    trait_descriptions = {
        "Openness": "curiosity, imagination, preference for novelty",
        "Conscientiousness": "organization, discipline, reliability",
        "Extraversion": "sociability, assertiveness, energy",
        "Agreeableness": "empathy, cooperation, kindness",
        "Neuroticism": "emotional instability, anxiety, mood swings"
    }

    return [
        {
            "role": "user",
            "content": (
                f'You are an expert in personality psychology using the Big Five model.\n\n'
                f'Determine whether the trait "{trait}" is present in the TEXT.\n\n'
                f'Trait definition:\n{trait_descriptions[trait]}\n\n'
                f'Guidelines:\n'
                f'- Look at tone, wording, emotional expression, and behavior\n'
                f'- Consider both explicit and implicit signals\n\n'
                f'Output:\n'
                f'- "1" if present\n'
                f'- "0" if absent\n'
                f'- Output ONLY one digit\n\n'
                f'TEXT:\n{text}'
            )
        }
    ]

  # enriched prompt

In [ ]:
def predict_mistral(text, trait, enriched=True):
    messages = build_mistral_enriched_prompt(text, trait) if enriched else build_mistral_simple_prompt(text, trait)

    # apply Mistral chat template
    encoded = mistral_tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    ).to(mistral_model.device)

    outputs = mistral_model.generate(
        encoded,
        max_new_tokens=5,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=mistral_tokenizer.eos_token_id
    )

    # decode only the newly generated tokens
    new_tokens = outputs[0][encoded.shape[-1]:]
    decoded = mistral_tokenizer.decode(new_tokens, skip_special_tokens=True)

    return parse_output(decoded)

  # mistral inference function

In [ ]:
def run_evaluation_mistral(dataset, enriched=True):
    results = {}

    for trait in traits:
        y_true = []
        y_pred = []

        for user in dataset:
            text = aggregate_user_comments(user["comments"])

            pred = predict_mistral(text, trait, enriched=enriched)

            if pred is None:
                continue

            y_true.append(user["labels"][trait])
            y_pred.append(pred)

        print(f"\n===== {trait} =====")
        print(classification_report(y_true, y_pred, digits=3))

        results[trait] = (y_true, y_pred)

    return results
   # dataset loop

In [ ]:
mistral_results = run_evaluation_mistral(dataset, enriched=True)

# evaluation

In [ ]:
for trait in mistral_results:
    y_true, y_pred = mistral_results[trait]

    print(f"\n{trait}")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("F1:", f1_score(y_true, y_pred))

# metrics